In [ ]:
from math import comb
from collections import defaultdict
import matplotlib.pyplot as plt
import numpy as np

def calculate_suit_distributions():
    """Calculate all possible suit distributions in a bridge hand."""
    distribution_counts = defaultdict(int)
    variant_stats = defaultdict(int)
    TOTAL_COMBINATIONS = comb(52, 13)
    
    print("Calculating all possible 13-card suit distributions...")
    
    for spades in range(14):
        for hearts in range(14):
            for diamonds in range(14):
                for clubs in range(14):
                    if (spades + hearts + diamonds + clubs) == 13:
                        current_dist = (spades, hearts, diamonds, clubs)
                        sorted_dist = tuple(sorted(current_dist, reverse=True))
                        combinations = (comb(13, spades) * comb(13, hearts) * 
                                       comb(13, diamonds) * comb(13, clubs))
                        distribution_counts[sorted_dist] += combinations
    
    return distribution_counts, TOTAL_COMBINATIONS

def analyze_variants(distribution_counts):
    """Analyze symmetry variants of each distribution."""
    variant_stats = defaultdict(int)
    variant_examples = {}
    
    for dist in distribution_counts:
        unique_lengths = len(set(dist))
        if unique_lengths == 4:   # All suits different (24 permutations)
            variants = 24
        elif unique_lengths == 3: # Two suits same (12 permutations)
            variants = 12
        elif unique_lengths == 2:
            if dist.count(dist[0]) == 2: # Two suits same, other two same (6 permutations)
                variants = 6
            else: # Three suits same (4 permutations)
                variants = 4
        else: # All four suits same (1 permutation)
            variants = 1
        
        variant_stats[variants] += 1
        if variants not in variant_examples:
            variant_examples[variants] = dist
    
    return variant_stats, variant_examples

def print_results(distribution_counts, variant_stats, TOTAL_COMBINATIONS):
    """Print the analysis results in a formatted way."""
    sorted_d = sorted(distribution_counts.items(), 
                     key=lambda kv: (-kv[1], kv[0]))
    
    print("\n" + "="*80)
    print(f"{'Rank':<5} {'Distribution':<15} {'Combinations':<20} {'Probability':<15}")
    print("-"*80)
    
    top_5_total = 0
    for i, (dist, count) in enumerate(sorted_d, 1):
        prob = count / TOTAL_COMBINATIONS * 100
        if i <= 5:
            top_5_total += count
        print(f"{i:<5} {str(dist):<15} {count:<20_} {prob:.6f}%")
    
    print("="*80)
    print(f"\nTop 5 distributions account for {top_5_total/TOTAL_COMBINATIONS*100:.2f}% of all hands")
    print(f"Total unique distributions: {len(distribution_counts)}")
    
    print("\nVariant Analysis:")
    print(f"{'Variants':<10} {'Count':<10} {'Example Distribution':<20}")
    print("-"*40)
    for variants, count in sorted(variant_stats.items(), reverse=True):
        example = next(d for d in distribution_counts if len(set(d)) == 
                      (4 if variants==24 else 3 if variants==12 else 
                       2 if variants in (6,4) else 1))
        print(f"{variants:<10} {count:<10} {str(example):<20}")

def plot_distributions(distribution_counts, TOTAL_COMBINATIONS):
    """Visualize the distribution probabilities."""
    sorted_d = sorted(distribution_counts.items(), 
                     key=lambda kv: (-kv[1], kv[0]))
    
    # Prepare data
    distributions = [f"{d[0]}-{d[1]}-{d[2]}-{d[3]}" for d, _ in sorted_d[:20]]
    probabilities = [c/TOTAL_COMBINATIONS*100 for _, c in sorted_d[:20]]
    
    # Create plot
    plt.figure(figsize=(12, 8))
    bars = plt.barh(distributions[::-1], probabilities[::-1], color='#1f77b4')
    plt.title('Top 20 Most Common Bridge Hand Distributions', pad=20)
    plt.xlabel('Probability (%)', labelpad=10)
    plt.ylabel('Suit Distribution (S-H-D-C)', labelpad=10)
    plt.grid(axis='x', linestyle='--', alpha=0.7)
    
    # Annotate bars with percentages
    for bar in bars:
        width = bar.get_width()
        plt.text(width + 0.1, bar.get_y() + bar.get_height()/2,
                f'{width:.2f}%', ha='left', va='center')
    
    plt.tight_layout()
    plt.savefig('bridge_distributions.png', dpi=300, bbox_inches='tight')
    plt.show()

def main():
    """Main function to run the analysis."""
    print("Bridge Hand Distribution Analyzer")
    print("Calculating probabilities for all possible suit distributions...\n")
    
    dist_counts, total = calculate_suit_distributions()
    var_stats, var_examples = analyze_variants(dist_counts)
    
    print_results(dist_counts, var_stats, total)
    plot_distributions(dist_counts, total)
    
    print("\nAnalysis complete! Visualization saved as 'bridge_distributions.png'")

if __name__ == "__main__":
    main()